# Aufgabe 1 – Ein- und Ausgabeformen einfacher RNNs

In dieser Aufgabe untersuchen Sie, wie ein RNN mehrdimensionale Zeitreihen
verarbeitet und warum unterschiedliche Zieldefinitionen zu unterschiedlichen
Ausgabeformen führen. Sie beginnen mit einer kleinen manuellen Rechnung und
entwickeln anschließend zwei eigene RNN-Modelle für synthetische Raumklimadaten.

Der Code enthält nur die Ausgangsdaten. Wie Sie die Berechnungen, Modelle und
Auswertungen strukturieren, entscheiden Sie selbst. Begründen Sie wichtige
Entscheidungen kurz im Markdown. Visualisierungen sind möglich, aber nicht
vorgegeben.


## Teil A – Einen RNN-Ablauf verstehen

Eine Sequenz besteht aus drei Zeitschritten mit jeweils zwei Merkmalen. Der
Hidden State hat zwei Komponenten; pro Zeitschritt entsteht ein Ausgabewert.
Für alle Zeitschritte werden dieselben Gewichte verwendet:

$$h_t = \tanh(W_x x_t + W_h h_{t-1} + b_h)$$

$$y_t = W_y h_t + b_y.$$

Bearbeiten Sie die folgenden Punkte:

1. Bestimmen Sie die Formen der vollständigen Sequenz, eines Zeitschritts, des
   Hidden States und aller Gewichtsmatrizen.
2. Berechnen Sie $h_1$ und $y_1$ zunächst von Hand.
3. Implementieren Sie danach den vollständigen Durchlauf über alle drei
   Zeitschritte. Speichern Sie sämtliche Hidden States und Ausgaben.
4. Vergleichen Sie den ersten berechneten Schritt mit Ihrer Handrechnung.

Dokumentieren Sie die Formen und die auf vier Nachkommastellen gerundeten Werte.


In [ ]:
import numpy as np
import tensorflow as tf

np.random.seed(42)
tf.keras.utils.set_random_seed(42)

x = np.array([[1.0, 0.5], [0.2, -0.1], [0.4, 0.8]])
W_x = np.array([[0.5, -0.2], [0.3, 0.8]])
W_h = np.array([[0.4, 0.1], [-0.5, 0.2]])
b_h = np.array([0.1, -0.1])
W_y = np.array([[0.7, -0.4]])
b_y = np.array([0.05])
h_0 = np.zeros(2)


### Verständnisfragen zum Hidden State

Beantworten Sie jeweils in zwei bis drei Sätzen:

1. Warum verändert sich der Hidden State, obwohl dieselben Gewichte in jedem
   Zeitschritt verwendet werden?
2. Welche Informationen gehen in die Berechnung von $h_3$ ein?
3. Was würde sich ändern, wenn der anfängliche Hidden State nicht null wäre?
4. Warum kann ein RNN pro Zeitschritt mehrere Merkmale gleichzeitig aufnehmen?
5. Worin unterscheidet sich das Teilen von Gewichten über die Zeit vom Teilen
   eines einzelnen Hidden States?


## Teil B – Raumklimadaten in Sequenzen umwandeln

Die nächste Zelle erzeugt eine reproduzierbare Zeitreihe aus Temperatur und
Luftfeuchtigkeit. Temperatur enthält periodische Muster und einen leichten Trend.
Die Luftfeuchtigkeit hängt teilweise von einer verzögerten Temperatur ab.

Bereiten Sie die Daten anschließend selbst für das Training vor:

1. Teilen Sie die Zeitreihe chronologisch in 70 % Training, 15 % Validierung und
   15 % Test.
2. Standardisieren Sie beide Merkmale. Mittelwert und Standardabweichung dürfen
   ausschließlich aus dem Trainingsabschnitt stammen.
3. Erzeugen Sie überlappende Fenster mit 24 Zeitschritten.
4. Erstellen Sie zu jedem Fenster zwei verschiedene Ziele:
   - eine um einen Schritt verschobene Temperatursequenz,
   - einen einzelnen Temperaturwert direkt nach dem Eingabefenster.
5. Prüfen und dokumentieren Sie die Formen aller erzeugten Arrays.

Die Fenster der drei Datenabschnitte sollen getrennt erzeugt werden. Kein Fenster
darf eine Grenze zwischen Training, Validierung und Test überschreiten.


In [ ]:
n = 1800
t = np.arange(n)
temperature = (
    20
    + 4 * np.sin(2 * np.pi * t / 48)
    + 1.5 * np.sin(2 * np.pi * t / 240)
    + 0.002 * t
    + np.random.normal(0, 0.35, n)
)
delayed_temperature = np.r_[np.repeat(temperature[0], 6), temperature[:-6]]
humidity = 58 - 0.7 * delayed_temperature + np.random.normal(0, 1.2, n)
raw_data = np.column_stack([temperature, humidity]).astype("float32")


### Fragen zur Datenvorbereitung

1. Welche Bedeutung besitzen die drei Dimensionen eines Eingabebatches?
2. Welche Form hat eine einzelne Sequenz und welche ein Batch aus 32 Sequenzen?
3. Warum wäre eine zufällige Aufteilung für diese Aufgabe problematisch?
4. Welche Information aus Validierung oder Test würde in das Training gelangen,
   wenn die Standardisierung mit dem vollständigen Datensatz berechnet würde?
5. Warum werden Temperatur und Luftfeuchtigkeit innerhalb eines Zeitschritts
   gemeinsam und nicht nacheinander eingelesen?
6. Weshalb ist der einzelne Zielwert identisch mit der letzten Position der
   verschobenen Zielsequenz?


## Teil C – Ein Sequence-to-Sequence-Modell entwickeln

Entwickeln Sie ein einfaches RNN, das zu jeder Position des Eingabefensters die
Temperatur des folgenden Zeitschritts vorhersagt. Die Architektur soll klein
bleiben, ihre konkrete Größe bestimmen Sie jedoch selbst.

Halten Sie vor der Implementierung fest:

- welche Eingabe- und Ausgabeform benötigt wird,
- wie viele rekurrente Schichten und Hidden Units Sie verwenden,
- welche Einstellung von `return_sequences` zur Zieldefinition passt,
- welche Ausgabeschicht, Aktivierung und Loss-Funktion für Regression geeignet
  sind.

Trainieren Sie mit Early Stopping und werten Sie das Modell auf dem Testabschnitt
in Grad Celsius aus. Berichten Sie mindestens MAE, Parameterzahl und tatsächlich
trainierte Epochen. Untersuchen Sie außerdem, ob sich der Fehler zwischen frühen
und späten Positionen eines Fensters unterscheidet. Wie Sie das untersuchen,
entscheiden Sie selbst.


## Teil D – Ein Sequence-to-Vector-Modell entwickeln

Entwickeln Sie nun ein zweites RNN, das aus demselben 24er-Fenster nur die
Temperatur direkt nach dem Fenster vorhersagt. Wählen und begründen Sie erneut
Architekturgröße, `return_sequences`, Ausgabeschicht und Trainingsparameter.

Verwenden Sie für beide Modelle möglichst vergleichbare Trainingsbedingungen.
Berichten Sie Test-MAE, Parameterzahl und Epochen. Prüfen Sie zusätzlich, ob die
Vorhersagen systematisch zu hoch oder zu niedrig liegen.


## Teil E – Ergebnisse vergleichen und einordnen

Fassen Sie beide Modelle in einer selbst erstellten Tabelle zusammen. Vergleichen
Sie Eingabeform, Ausgabeform, Zieldefinition, `return_sequences`, Parameterzahl
und Fehlermaß.

Beantworten Sie abschließend:

1. Warum können beide Modelle dieselbe Eingabeform, aber unterschiedliche
   Ausgabeformen besitzen?
2. Welche Bedeutung hat der letzte Hidden State im Sequence-to-Vector-Modell?
3. Warum sind die beiden globalen MAE-Werte nicht vollständig gleichartig?
4. Für welche reale Aufgabe wäre Sequence-to-Sequence sinnvoll, für welche
   Sequence-to-Vector?
5. Welche zusätzliche Information könnte die Luftfeuchtigkeit liefern?
6. Welche Änderung am Experiment würde zeigen, ob das zweite Merkmal wirklich
   hilft?
7. Welche Schlussfolgerung ist durch Ihre Messwerte belegt und welche wäre nur
   eine Vermutung?
